# LLMOps: API Cost, Latency, and Prompt Version Management
**Course:** MLOps  |  **Experiential Learning Assignment-2**  |  **Scenario:** EduAssist AI (academic question-answering assistant)

This notebook covers all three tasks of the assignment:

| Task | Marks | Section |
|---|---|---|
| 1. Call an LLM and measure latency | 2 | Task 1 |
| 2. Count tokens and calculate cost | 3 | Task 2 |
| 3. Prompt versioning and MLflow evaluation | 5 | Task 3 |

**Model used.** API access is not used; as the assignment allows, a locally hosted open-source model (`Qwen/Qwen2.5-1.5B-Instruct`, run with Hugging Face `transformers` on CPU) is used instead. Token counts come from the model's own tokenizer, and cost is estimated from a published hosting price (see the config cell). No API key is required; an optional `HF_TOKEN` environment variable is read only if set (it is never hard-coded).

## 0. Setup

In [1]:
# Install the required libraries only if something is missing (see requirements.txt).
import importlib.util, subprocess, sys
_needed = ["mlflow", "transformers", "torch", "pandas"]
if any(importlib.util.find_spec(m) is None for m in _needed):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("Dependencies installed.")
else:
    print("All required libraries are already installed.")

All required libraries are already installed.


In [2]:
import os, json, time, platform, difflib
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch, transformers, mlflow
from transformers import AutoTokenizer, AutoModelForCausalLM
from mlflow.entities import AssessmentSource, AssessmentSourceType

# ---------------- Configuration (everything that may change is here) ----------------
MODEL_ID = os.environ.get("EDUASSIST_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
HF_TOKEN = os.environ.get("HF_TOKEN")      # optional; read from the environment, never hard-coded
MAX_NEW_TOKENS = 256                       # same cap for every request
DECODING = "greedy (do_sample=False, i.e. temperature 0)"

# Published hosting price used to ESTIMATE cost of the locally hosted model (no real bill exists).
PRICE_INPUT_PER_M = 0.10                   # USD per 1,000,000 input tokens
PRICE_OUTPUT_PER_M = 0.10                  # USD per 1,000,000 output tokens
CURRENCY = "USD"
PRICING_DATE = "retrieved 2026-10-31 (source page last updated 2026-07-21)"
PRICING_SOURCE = ("Hosted-API list price for Qwen2.5-1.5B-Instruct: "
                  "https://pricepertoken.com/pricing-page/model/qwen-qwen2.5-1.5b-instruct "
                  "(cross-checked with the Fireworks listing on getmaxim.ai: $0.10 in / $0.10 out per 1M tokens)")
REQUESTS_PER_MONTH = 10_000

EXPERIMENT_NAME = "eduassist_llmops_assignment"
PROMPT_NAME = "eduassist_qa_prompt"        # one registry entry holding versions 1, 2, 3
TRACKING_URI = f"sqlite:///{Path('mlflow.db').resolve()}"

torch.set_num_threads(os.cpu_count() or 1)
if MODEL_ID != "Qwen/Qwen2.5-1.5B-Instruct":
    print("WARNING: model changed - update the price settings above to match the new model.")

print("Python        :", platform.python_version())
print("torch         :", torch.__version__, "| transformers:", transformers.__version__, "| mlflow:", mlflow.__version__)
print("pandas        :", pd.__version__, "| CPU threads:", torch.get_num_threads(), "| CUDA:", torch.cuda.is_available())
print("Model         :", MODEL_ID, "| decoding:", DECODING, "| max_new_tokens:", MAX_NEW_TOKENS)
print(f"Pricing       : {PRICE_INPUT_PER_M} {CURRENCY}/M input, {PRICE_OUTPUT_PER_M} {CURRENCY}/M output, {PRICING_DATE}")
print("Pricing source:", PRICING_SOURCE)

/home/student/workspace/eduassist-llmops-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python        : 3.12.9
torch         : 2.15.0+cpu | transformers: 5.22.1 | mlflow: 3.17.2
pandas        : 3.1.0 | CPU threads: 4 | CUDA: False
Model         : Qwen/Qwen2.5-1.5B-Instruct | decoding: greedy (do_sample=False, i.e. temperature 0) | max_new_tokens: 256
Pricing       : 0.1 USD/M input, 0.1 USD/M output, retrieved 2026-10-31 (source page last updated 2026-07-21)
Pricing source: Hosted-API list price for Qwen2.5-1.5B-Instruct: https://pricepertoken.com/pricing-page/model/qwen-qwen2.5-1.5b-instruct (cross-checked with the Fireworks listing on getmaxim.ai: $0.10 in / $0.10 out per 1M tokens)


### Questions and prompt versions
Five academic questions (the same five are used in every task) and the three prompt versions exactly as given in the assignment.

In [3]:
QUESTIONS = [
    ("Q1", "What is overfitting in machine learning?"),
    ("Q2", "Explain the difference between supervised and unsupervised learning."),
    ("Q3", "What is gradient descent and why is it used?"),
    ("Q4", "What is the bias-variance tradeoff?"),
    ("Q5", "What is the difference between precision and recall?"),
]

V1_TEMPLATE = """Answer the following student question clearly.
Question: {{question}}"""

V2_TEMPLATE = """You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.
Question: {{question}}"""

V3_TEMPLATE = """You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}"""

print(len(QUESTIONS), "questions loaded")

5 questions loaded


### Load the model and define the LLM call helper
The helper records start/end time, latency, response status and token usage for every request. Model loading and one short warm-up generation happen **before** any measurement, so one-off start-up cost does not distort latency.

In [4]:
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=HF_TOKEN, dtype=torch.float32)
except TypeError:                                    # older transformers versions
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=HF_TOKEN, torch_dtype=torch.float32)
model.eval()
print(f"Model loaded in {time.perf_counter() - t0:.1f}s | revision: {getattr(model.config, '_commit_hash', 'n/a')}")


def call_llm(prompt_text):
    """Send one prompt to the local model; return text, token usage, timing and status."""
    start = datetime.now()
    t_start = time.perf_counter()
    text, n_in, n_out, status = "", 0, 0, "success"
    try:
        chat = [{"role": "user", "content": prompt_text}]
        rendered = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
        n_in = int(inputs["input_ids"].shape[1])           # input tokens (includes chat-template tokens)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        new_ids = out[0][n_in:]
        n_out = int(new_ids.shape[0])                      # output tokens
        text = tokenizer.decode(new_ids, skip_special_tokens=True)
    except Exception as e:                                 # a failed request is recorded, not fatal
        status = f"error: {type(e).__name__}: {e}"
    t_end = time.perf_counter()
    return {"response_text": text, "input_tokens": n_in, "output_tokens": n_out,
            "total_tokens": n_in + n_out, "status": status,
            "start_time": start.isoformat(timespec="milliseconds"),
            "end_time": datetime.now().isoformat(timespec="milliseconds"),
            "latency_seconds": t_end - t_start}


# warm-up (not recorded)
_ = call_llm("Say hello.")
print("Warm-up done.")

Loading weights:   0%|                                                                         | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|▏                                                                | 1/338 [00:00<01:23,  4.02it/s]

Loading weights:  30%|██████████████████▎                                           | 100/338 [00:00<00:00, 341.15it/s]

Loading weights:  44%|███████████████████████████▌                                  | 150/338 [00:00<00:00, 323.98it/s]

Loading weights:  58%|████████████████████████████████████▏                         | 197/338 [00:00<00:00, 325.11it/s]

Loading weights:  72%|████████████████████████████████████████████▌                 | 243/338 [00:00<00:00, 350.71it/s]

Loading weights:  84%|███████████████████████████████████████████████████▉          | 283/338 [00:00<00:00, 318.66it/s]

Loading weights:  96%|███████████████████████████████████████████████████████████▏  | 323/338 [00:01<00:00, 338.97it/s]

Loading weights: 100%|██████████████████████████████████████████████████████████████| 338/338 [00:01<00:00, 315.47it/s]

Model loaded in 29.1s | revision: n/a


Warm-up done.


## Task 1: Call an LLM and Measure Latency (2 marks)
Five academic questions are sent to the model with **Prompt v1 (basic)**. For each request the question ID, prompt version, model name, start time, end time, latency and response status are recorded.

In [5]:
rows = []
for qid, question in QUESTIONS:
    r = call_llm(V1_TEMPLATE.replace("{{question}}", question))
    rows.append({"question_id": qid, "prompt_version": "v1", "model": MODEL_ID, **r})
    print(f"{qid}: {r['latency_seconds']:.2f}s | in={r['input_tokens']} out={r['output_tokens']} | {r['status']}")

task1 = pd.DataFrame(rows)
task1["latency_seconds"] = task1["latency_seconds"].round(3)
print()
display(task1[["question_id", "prompt_version", "model", "start_time", "end_time", "latency_seconds", "status"]])

Q1: 116.13s | in=47 out=218 | success


Q2: 103.42s | in=49 out=174 | success


Q3: 120.52s | in=48 out=234 | success


Q4: 127.30s | in=47 out=256 | success


Q5: 112.96s | in=47 out=223 | success



,question_id,prompt_version,model,start_time,end_time,latency_seconds,status
0,Q1,v1,Qwen/Qwen2.5-1.5B-Instruct,2026-10-31T12:51:24.849,2026-10-31T12:53:20.976,116.126,success
1,Q2,v1,Qwen/Qwen2.5-1.5B-Instruct,2026-10-31T12:53:20.976,2026-10-31T12:55:04.400,103.424,success
2,Q3,v1,Qwen/Qwen2.5-1.5B-Instruct,2026-10-31T12:55:04.400,2026-10-31T12:57:04.919,120.519,success
3,Q4,v1,Qwen/Qwen2.5-1.5B-Instruct,2026-10-31T12:57:04.919,2026-10-31T12:59:12.224,127.305,success
4,Q5,v1,Qwen/Qwen2.5-1.5B-Instruct,2026-10-31T12:59:12.224,2026-10-31T13:01:05.183,112.958,success


In [6]:
ok = task1[task1["status"] == "success"]["latency_seconds"]
if len(ok) < len(task1):
    print(f"NOTE: {len(task1) - len(ok)} request(s) failed and are excluded from the statistics.")

latency_summary = pd.Series({
    "mean": ok.mean(), "median": ok.median(), "min": ok.min(),
    "max": ok.max(), "std (sample)": ok.std(ddof=1),
}, name="latency_seconds").round(3)
display(latency_summary.to_frame())

,latency_seconds
mean,116.066
median,116.126
min,103.424
max,127.305
std (sample),8.883


In [7]:
slow = task1.loc[task1["latency_seconds"].idxmax()]
r_corr = task1["latency_seconds"].corr(task1["output_tokens"])
hit_cap = int(slow["output_tokens"]) >= MAX_NEW_TOKENS

corr_text = ""
if pd.notna(r_corr):
    if r_corr >= 0.7:
        corr_text = (f"Across the five requests, latency and output length are strongly correlated (r = {r_corr:.2f}), which fits how an "
                     f"autoregressive model works: it produces one token at a time, so longer answers take longer, and on a CPU-only "
                     f"machine this per-token cost is high. ")
    else:
        corr_text = (f"Across the five requests, latency and output length are only weakly correlated (r = {r_corr:.2f}), so output length "
                     f"alone does not explain the differences. ")

interpretation = (
    f"The slowest request was {slow['question_id']} at {slow['latency_seconds']:.2f} s, against a mean of "
    f"{ok.mean():.2f} s and a fastest request of {ok.min():.2f} s. It generated {int(slow['output_tokens'])} output tokens"
    f"{' (it reached the max_new_tokens cap of ' + str(MAX_NEW_TOKENS) + ')' if hit_cap else ''}, "
    f"while the average request generated {task1['output_tokens'].mean():.0f}. "
    + corr_text +
    f"Other possible contributors are a longer prompt (more input tokens to process) and run-to-run variation from other "
    f"processes sharing the same CPU."
)
print(interpretation)

The slowest request was Q4 at 127.31 s, against a mean of 116.07 s and a fastest request of 103.42 s. It generated 256 output tokens (it reached the max_new_tokens cap of 256), while the average request generated 221. Across the five requests, latency and output length are strongly correlated (r = 0.97), which fits how an autoregressive model works: it produces one token at a time, so longer answers take longer, and on a CPU-only machine this per-token cost is high. Other possible contributors are a longer prompt (more input tokens to process) and run-to-run variation from other processes sharing the same CPU.


## Task 2: Count Tokens and Calculate Cost (3 marks)
Token usage is taken from each response (input tokens from the tokenizer, output tokens from the generated ids). Cost uses the assignment formulas with the published hosting price from the config cell:

- Input Cost = Input Tokens / 1,000,000 × Input Price per Million Tokens
- Output Cost = Output Tokens / 1,000,000 × Output Price per Million Tokens
- Total Cost per Request = Input Cost + Output Cost
- Projected Monthly Cost = Average Cost per Request × 10,000

*A locally hosted model produces no real bill, so the cost is an estimate based on the published hosting price of the same model (see the config cell for the date, currency and source).*

In [8]:
task2 = task1.copy()
task2["input_cost"]  = task2["input_tokens"]  / 1_000_000 * PRICE_INPUT_PER_M
task2["output_cost"] = task2["output_tokens"] / 1_000_000 * PRICE_OUTPUT_PER_M
task2["total_cost"]  = task2["input_cost"] + task2["output_cost"]

cost_table = task2[["question_id", "input_tokens", "output_tokens", "total_tokens",
                    "input_cost", "output_cost", "total_cost"]].copy()
cost_table = cost_table.rename(columns={"input_cost": f"input_cost_{CURRENCY}",
                                        "output_cost": f"output_cost_{CURRENCY}",
                                        "total_cost": f"total_cost_{CURRENCY}"})
display(cost_table.style.format({f"input_cost_{CURRENCY}": "{:.8f}", f"output_cost_{CURRENCY}": "{:.8f}",
                                 f"total_cost_{CURRENCY}": "{:.8f}"}))

avg_cost = task2["total_cost"].mean()
monthly = avg_cost * REQUESTS_PER_MONTH
print(f"Average cost per request       : {avg_cost:.8f} {CURRENCY}")
print(f"Projected monthly cost ({REQUESTS_PER_MONTH:,} requests): {monthly:.4f} {CURRENCY}")

,question_id,input_tokens,output_tokens,total_tokens,input_cost_USD,output_cost_USD,total_cost_USD
0,Q1,47,218,265,0.00000470,0.00002180,0.00002650
1,Q2,49,174,223,0.00000490,0.00001740,0.00002230
2,Q3,48,234,282,0.00000480,0.00002340,0.00002820
3,Q4,47,256,303,0.00000470,0.00002560,0.00003030
4,Q5,47,223,270,0.00000470,0.00002230,0.00002700


Average cost per request       : 0.00002686 USD
Projected monthly cost (10,000 requests): 0.2686 USD


In [9]:
# Cost-optimisation recommendation: shorten responses (output tokens are generated one by one and, here, billed at the same rate as input).
out_share = task2["output_cost"].sum() / task2["total_cost"].sum()
CAP = 128
capped_out = task2["output_tokens"].clip(upper=CAP)
capped_cost = (task2["input_tokens"] / 1e6 * PRICE_INPUT_PER_M + capped_out / 1e6 * PRICE_OUTPUT_PER_M).mean()
saving = 1 - capped_cost / avg_cost

recommendation = (
    f"Recommendation - shorten responses. Output tokens make up {out_share:.0%} of the cost of these requests "
    f"(average {task2['output_tokens'].mean():.0f} output vs {task2['input_tokens'].mean():.0f} input tokens). "
    f"Asking for concise answers (Prompt v2 does this) and capping output length would reduce both cost and latency; "
    f"as an illustration, capping each answer at {CAP} tokens would lower the average cost per request from "
    f"{avg_cost:.8f} to {capped_cost:.8f} {CURRENCY} (about {saving:.0%}), i.e. the monthly projection from "
    f"{monthly:.4f} to {capped_cost * REQUESTS_PER_MONTH:.4f} {CURRENCY}. The trade-off is that a hard cap can cut answers "
    f"short, so it should be checked against answer quality."
)
print(recommendation)

Recommendation - shorten responses. Output tokens make up 82% of the cost of these requests (average 221 output vs 48 input tokens). Asking for concise answers (Prompt v2 does this) and capping output length would reduce both cost and latency; as an illustration, capping each answer at 128 tokens would lower the average cost per request from 0.00002686 to 0.00001756 USD (about 35%), i.e. the monthly projection from 0.2686 to 0.1756 USD. The trade-off is that a hard cap can cut answers short, so it should be checked against answer quality.


In [10]:
# Save the Task 1 + Task 2 results.
csv1 = task2[["question_id", "prompt_version", "model", "start_time", "end_time", "latency_seconds", "status",
              "input_tokens", "output_tokens", "total_tokens", "input_cost", "output_cost", "total_cost"]].copy()
csv1 = csv1.rename(columns={"input_cost": f"input_cost_{CURRENCY}", "output_cost": f"output_cost_{CURRENCY}",
                            "total_cost": f"total_cost_{CURRENCY}"})
csv1.to_csv("latency_tokens_cost.csv", index=False)
print("Saved latency_tokens_cost.csv", csv1.shape)

Saved latency_tokens_cost.csv (5, 13)


## Task 3: Prompt Versioning and MLflow Evaluation (5 marks)
1. Create the MLflow experiment `eduassist_llmops_assignment`.
2. Register Prompt v1, v2 and v3 in the MLflow Prompt Registry (one prompt, three versions, each with a commit message).
3. Run every version on the same five questions and log `avg_latency_seconds`, `avg_input_tokens`, `avg_output_tokens`, `avg_cost_per_request` and `valid_json`.
4. Compare the versions and promote the best one with the `production` alias.

In [11]:
mlflow.set_tracking_uri(TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print("Tracking URI :", TRACKING_URI)
print("Experiment   :", experiment.name, "| id:", experiment.experiment_id)

2026/10/31 13:01:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/31 13:01:09 INFO mlflow.store.db.utils: Updating database tables


2026/10/31 13:01:40 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


Tracking URI : sqlite:////home/student/workspace/eduassist-llmops-lab/mlflow.db
Experiment   : eduassist_llmops_assignment | id: 1


### 3.1 Register the three prompt versions

In [12]:
TEMPLATES = {1: V1_TEMPLATE, 2: V2_TEMPLATE, 3: V3_TEMPLATE}
COMMIT_MESSAGES = {
    1: "v1 basic: one-line instruction, no role, no length or format constraints (baseline)",
    2: "v2 structured: adds academic-assistant role, concise and accurate tone, simple language for undergraduates",
    3: "v3 json output: strict JSON-only reply with fixed schema (answer, confidence, key_points); no markdown or code fences",
}

def _already_registered(version):
    try:
        return mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{version}")
    except Exception:
        return None

for v in (1, 2, 3):
    existing = _already_registered(v)
    if existing is not None:                       # makes the notebook safe to re-run
        print(f"v{v} already registered (version {existing.version}) - reusing it")
        continue
    pv = mlflow.genai.register_prompt(name=PROMPT_NAME, template=TEMPLATES[v], commit_message=COMMIT_MESSAGES[v])
    print(f"Registered {PROMPT_NAME} version {pv.version}: {COMMIT_MESSAGES[v]}")

Registered eduassist_qa_prompt version 1: v1 basic: one-line instruction, no role, no length or format constraints (baseline)


Registered eduassist_qa_prompt version 2: v2 structured: adds academic-assistant role, concise and accurate tone, simple language for undergraduates


Registered eduassist_qa_prompt version 3: v3 json output: strict JSON-only reply with fixed schema (answer, confidence, key_points); no markdown or code fences


### 3.2 Run each prompt version on the same five questions
Every request is traced, and a boolean `valid_json` feedback (does `json.loads` accept the response exactly as returned?) is attached to each trace. MLflow averages boolean feedback, which is the JSON-validity pass rate:

valid_json = Valid JSON Responses / Total Responses

A response wrapped in a markdown code fence or followed by extra text is counted as **not valid**. `schema_ok` (extra, for the quality discussion) additionally checks the exact schema: keys `answer` (string), `confidence` (high/medium/low) and `key_points` (list of strings).

In [13]:
def check_json(text):
    """Return (valid_json, schema_ok) for one raw model response."""
    try:
        obj = json.loads(text.strip())
    except Exception:
        return False, False
    schema_ok = (isinstance(obj, dict)
                 and set(obj.keys()) == {"answer", "confidence", "key_points"}
                 and isinstance(obj["answer"], str)
                 and obj["confidence"] in {"high", "medium", "low"}
                 and isinstance(obj["key_points"], list)
                 and all(isinstance(k, str) for k in obj["key_points"]))
    return True, schema_ok


@mlflow.trace(name="eduassist_qa")
def answer_question(prompt_version, question_id, prompt_text):
    mlflow.update_current_trace(tags={"prompt_version": prompt_version, "question_id": question_id})
    return call_llm(prompt_text)

In [14]:
eval_rows, version_metrics = [], {}

for v in (1, 2, 3):
    label = f"v{v}"
    with mlflow.start_run(run_name=f"prompt_{label}"):
        prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{v}")      # loaded inside the run, so MLflow links it
        mlflow.log_params({"prompt_name": PROMPT_NAME, "prompt_version": label, "model": MODEL_ID,
                           "max_new_tokens": MAX_NEW_TOKENS, "decoding": DECODING,
                           "price_input_per_M": PRICE_INPUT_PER_M, "price_output_per_M": PRICE_OUTPUT_PER_M,
                           "currency": CURRENCY})
        trace_ids, run_rows = [], []
        for qid, question in QUESTIONS:
            res = answer_question(label, qid, prompt.format(question=question))
            trace_ids.append(mlflow.get_last_active_trace_id())
            valid, schema_ok = check_json(res["response_text"])
            cost = (res["input_tokens"] / 1e6 * PRICE_INPUT_PER_M) + (res["output_tokens"] / 1e6 * PRICE_OUTPUT_PER_M)
            run_rows.append({"question_id": qid, "prompt_version": label, "model": MODEL_ID,
                             "latency_seconds": round(res["latency_seconds"], 3), "status": res["status"],
                             "input_tokens": res["input_tokens"], "output_tokens": res["output_tokens"],
                             "total_tokens": res["total_tokens"], f"total_cost_{CURRENCY}": cost,
                             "valid_json": valid, "schema_ok": schema_ok, "response_text": res["response_text"]})
            print(f"{label} {qid}: {res['latency_seconds']:.1f}s | out={res['output_tokens']} | valid_json={valid}")

        # per-response boolean feedback -> MLflow aggregates it into the pass rate
        mlflow.flush_trace_async_logging()
        for tid, row in zip(trace_ids, run_rows):
            mlflow.log_feedback(trace_id=tid, name="valid_json", value=bool(row["valid_json"]),
                                source=AssessmentSource(source_type=AssessmentSourceType.CODE, source_id="json.loads"))

        d = pd.DataFrame(run_rows)
        good = d[d["status"] == "success"]
        version_metrics[label] = {
            "avg_latency_seconds": good["latency_seconds"].mean(),
            "avg_input_tokens": good["input_tokens"].mean(),
            "avg_output_tokens": good["output_tokens"].mean(),
            "avg_cost_per_request": good[f"total_cost_{CURRENCY}"].mean(),
            "valid_json": d["valid_json"].mean(),            # all responses count; failed requests are invalid
            "schema_ok": d["schema_ok"].mean(),
        }
        for name in ("avg_latency_seconds", "avg_input_tokens", "avg_output_tokens", "avg_cost_per_request", "valid_json"):
            mlflow.log_metric(name, float(version_metrics[label][name]))
        eval_rows.extend(run_rows)
    print(f"-> {label} metrics logged: " + ", ".join(f"{k}={val:.4g}" for k, val in version_metrics[label].items()), "\n")

v1 Q1: 112.2s | out=218 | valid_json=False


v1 Q2: 90.6s | out=174 | valid_json=False


v1 Q3: 113.2s | out=234 | valid_json=False


v1 Q4: 118.7s | out=256 | valid_json=False


v1 Q5: 105.0s | out=223 | valid_json=False


-> v1 metrics logged: avg_latency_seconds=107.9, avg_input_tokens=47.6, avg_output_tokens=221, avg_cost_per_request=2.686e-05, valid_json=0, schema_ok=0 



... (8 outputs left out)

v3 Q3: 61.9s | out=116 | valid_json=False


v3 Q4: 81.3s | out=156 | valid_json=False


v3 Q5: 47.5s | out=88 | valid_json=False


-> v3 metrics logged: avg_latency_seconds=63.69, avg_input_tokens=103.6, avg_output_tokens=121, avg_cost_per_request=2.246e-05, valid_json=0, schema_ok=0 



### 3.3 Compare the versions and choose the best one

In [15]:
metrics = pd.DataFrame(version_metrics).T
# Selection rule: highest valid_json, then highest schema_ok, then lowest average latency.
order = metrics.sort_values(["valid_json", "schema_ok", "avg_latency_seconds"], ascending=[False, False, True])
best = order.index[0]
if metrics.loc[best, "valid_json"] == 0:
    print("WARNING: no prompt version produced valid JSON - review the responses before promoting anything.")

comparison = pd.DataFrame({
    "Prompt Version": metrics.index,
    "Avg Latency": metrics["avg_latency_seconds"].map("{:.2f} s".format).values,
    "Avg Input Tokens": metrics["avg_input_tokens"].map("{:.1f}".format).values,
    "Avg Output Tokens": metrics["avg_output_tokens"].map("{:.1f}".format).values,
    "Avg Cost": metrics["avg_cost_per_request"].map(lambda x: f"{x:.8f} {CURRENCY}").values,
    "Valid JSON Rate": metrics["valid_json"].map("{:.0%}".format).values,
    "Recommended?": ["Yes" if v == best else "No" for v in metrics.index],
})
display(comparison)
print("Schema-compliance rate (extra check):", {k: f"{v:.0%}" for k, v in metrics["schema_ok"].items()})
print("Best version by the selection rule  :", best)

,Prompt Version,Avg Latency,Avg Input Tokens,Avg Output Tokens,Avg Cost,Valid JSON Rate,Recommended?
0,v1,107.93 s,47.6,221.0,0.00002686 USD,0%,No
1,v2,92.71 s,69.6,193.8,0.00002634 USD,0%,No
2,v3,63.69 s,103.6,121.0,0.00002246 USD,0%,Yes


Schema-compliance rate (extra check): {'v1': '0%', 'v2': '0%', 'v3': '0%'}
Best version by the selection rule  : v3


In [16]:
# Prompt diff evidence (the MLflow UI shows the same diff: Prompts > eduassist_qa_prompt > Compare).
def show_diff(a, b):
    print(f"--- {a} -> {b} ---")
    for line in difflib.unified_diff(TEMPLATES[int(a[1])].splitlines(), TEMPLATES[int(b[1])].splitlines(),
                                     fromfile=a, tofile=b, lineterm=""):
        print(line)
    print()
show_diff("v1", "v2")
show_diff("v2", "v3")

--- v1 -> v2 ---
--- v1
+++ v2
@@ -1,2 +1,4 @@
-Answer the following student question clearly.
+You are an academic assistant.
+Answer the following student question in a clear, concise, and accurate manner.
+Use simple language suitable for undergraduate students.
 Question: {{question}}

--- v2 -> v3 ---
--- v2
+++ v3
@@ -1,4 +1,10 @@
 You are an academic assistant.
-Answer the following student question in a clear, concise, and accurate manner.
-Use simple language suitable for undergraduate students.
+Answer the student question and return ONLY valid JSON.
+Do not include markdown, explanations outside JSON, or code fences.
+Use this exact JSON schema:
+{
+  "answer": "string",
+  "confidence": "high | medium | low",
+  "key_points": ["string"]
+}
 Question: {{question}}



### 3.4 Promote the best version with the `production` alias

In [17]:
best_version = int(best[1])
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=best_version)

prod = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")      # verify by loading through the alias
print(f"Alias 'production' now points to {PROMPT_NAME} version {prod.version}")
print("Commit message:", prod.commit_message)
print("--- template ---")
print(prod.template)
assert prod.version == best_version

Alias 'production' now points to eduassist_qa_prompt version 3
Commit message: v3 json output: strict JSON-only reply with fixed schema (answer, confidence, key_points); no markdown or code fences
--- template ---
You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}


In [18]:
# Save per-response evaluation results.
results = pd.DataFrame(eval_rows)
results.to_csv("prompt_evaluation_results.csv", index=False)
print("Saved prompt_evaluation_results.csv", results.shape)
display(results[["prompt_version", "question_id", "latency_seconds", "input_tokens", "output_tokens",
                 "valid_json", "schema_ok"]])

Saved prompt_evaluation_results.csv (15, 12)


,prompt_version,question_id,latency_seconds,input_tokens,output_tokens,valid_json,schema_ok
0,v1,Q1,112.169,47,218,False,False
1,v1,Q2,90.594,49,174,False,False
2,v1,Q3,113.164,48,234,False,False
3,v1,Q4,118.745,47,256,False,False
4,v1,Q5,104.979,47,223,False,False
5,v2,Q1,60.876,69,124,False,False
6,v2,Q2,72.748,71,152,False,False
7,v2,Q3,119.711,70,256,False,False
8,v2,Q4,114.438,69,234,False,False
9,v2,Q5,95.756,69,203,False,False


### 3.5 Promotion justification

In [19]:
m, o = metrics, metrics.drop(index=best)
best_valid = m.loc[best, "valid_json"]
decisive = best_valid > o["valid_json"].max()

def cmp(col):
    """Describe how the promoted version compares with the others on one metric."""
    parts = []
    for other in o.index:
        a, b = m.loc[best, col], m.loc[other, col]
        parts.append(("lower than " if a < b else "higher than " if a > b else "equal to ") + other)
    return " and ".join(parts)

others_valid = ", ".join(f"{k} {m.loc[k, 'valid_json']:.0%}" for k in o.index)
if decisive:
    reliability = (
        f"The decisive factor is output reliability: the EduAssist application needs strict JSON (answer, confidence, key_points) that "
        f"downstream code can parse, and {best} achieved a valid_json rate of {best_valid:.0%} against {others_valid}. "
        f"Its exact-schema compliance was {m.loc[best, 'schema_ok']:.0%}, so the answers are not only parseable but also carry the "
        f"required fields, which is the quality dimension that matters for the product. ")
elif best_valid > 0:
    reliability = (
        f"{best} reached a valid_json rate of {best_valid:.0%}, which is not higher than the others ({others_valid}), so the exact-schema "
        f"compliance ({m.loc[best, 'schema_ok']:.0%}) and then latency decided the ranking. ")
else:
    reliability = (
        f"No version produced valid JSON (all valid_json rates are 0%), so this promotion is provisional: the ranking fell back to "
        f"latency and the responses should be reviewed before production use. ")

cheapest, fastest = m["avg_cost_per_request"].idxmin(), m["avg_latency_seconds"].idxmin()
tradeoff = []
if best != cheapest:
    extra = m.loc[best, "avg_cost_per_request"] / m.loc[cheapest, "avg_cost_per_request"] - 1
    tradeoff.append(f"costs {extra:.0%} more per request than {cheapest}, the cheapest version")
if best != fastest:
    extra = m.loc[best, "avg_latency_seconds"] / m.loc[fastest, "avg_latency_seconds"] - 1
    tradeoff.append(f"is {extra:.0%} slower than {fastest}, the fastest version")
tradeoff_text = (f"The trade-off of this choice is that {best} " + " and ".join(tradeoff) + "; this is accepted because "
                 + ("reliable machine-readable output is the stated requirement" if decisive else "no other version was better on the selection rule")
                 + ". ") if tradeoff else f"{best} is also the cheapest and the fastest of the three, so there is no trade-off. "

justification = (
    f"Prompt {best} is promoted to production. " + reliability +
    f"On latency, {best} averaged {m.loc[best, 'avg_latency_seconds']:.2f} s per request, which is {cmp('avg_latency_seconds')}; "
    f"latency mainly follows output length because the model generates one token at a time. "
    f"On tokens, {best} used {m.loc[best, 'avg_input_tokens']:.1f} input tokens ({cmp('avg_input_tokens')}) and produced "
    f"{m.loc[best, 'avg_output_tokens']:.1f} output tokens ({cmp('avg_output_tokens')}). "
    f"On cost, the estimated average is {m.loc[best, 'avg_cost_per_request']:.8f} {CURRENCY} per request ({cmp('avg_cost_per_request')}), "
    f"which projects to {m.loc[best, 'avg_cost_per_request'] * REQUESTS_PER_MONTH:.4f} {CURRENCY} for {REQUESTS_PER_MONTH:,} requests a month. "
    + tradeoff_text +
    f"The selection rule was explicit and repeatable (highest valid_json, then highest schema compliance, then lowest latency), "
    f"and the alias keeps the change reversible, because production can be pointed back to an earlier version in one call."
)
n_words = len(justification.split())
print(justification)
print(f"\n[{n_words} words]")
assert n_words >= 150, "justification must be at least 150 words"

Prompt v3 is promoted to production. No version produced valid JSON (all valid_json rates are 0%), so this promotion is provisional: the ranking fell back to latency and the responses should be reviewed before production use. On latency, v3 averaged 63.69 s per request, which is lower than v1 and lower than v2; latency mainly follows output length because the model generates one token at a time. On tokens, v3 used 103.6 input tokens (higher than v1 and higher than v2) and produced 121.0 output tokens (lower than v1 and lower than v2). On cost, the estimated average is 0.00002246 USD per request (lower than v1 and lower than v2), which projects to 0.2246 USD for 10,000 requests a month. v3 is also the cheapest and the fastest of the three, so there is no trade-off. The selection rule was explicit and repeatable (highest valid_json, then highest schema compliance, then lowest latency), and the alias keeps the change reversible, because production can be pointed back to an earlier version

### 3.6 Diagnosis: why is valid_json 0%?
The strict check treats a response wrapped in a markdown code fence as invalid. This cell re-reads the saved responses and shows what the rates would be if the fence were removed. It is an explanation only; the logged metric is unchanged.

In [21]:
import re
import numpy as np

def strip_fence(text):
    """Remove one surrounding markdown code fence (```json ... ```), if present."""
    m = re.fullmatch(r"```[a-zA-Z]*\s*(.*?)\s*```", text.strip(), flags=re.DOTALL)
    return m.group(1) if m else text.strip()

res = pd.read_csv("prompt_evaluation_results.csv")
res["response_text"] = res["response_text"].fillna("")
rows = []
for v, g in res.groupby("prompt_version"):
    after = [check_json(strip_fence(t)) for t in g["response_text"]]
    rows.append({
        "prompt_version": v,
        "valid_json (strict, as logged)": g["valid_json"].mean(),
        "responses starting with a code fence": g["response_text"].str.strip().str.startswith("```").mean(),
        "valid_json after removing the fence": np.mean([a for a, b in after]),
        "schema_ok after removing the fence": np.mean([b for a, b in after]),
    })
diagnosis = pd.DataFrame(rows).set_index("prompt_version")
display(diagnosis.style.format("{:.0%}"))
print("Strict valid_json is still the metric logged to MLflow; the table above only explains why it is 0%.")

,"valid_json (strict, as logged)",responses starting with a code fence,valid_json after removing the fence,schema_ok after removing the fence
prompt_version,,,,
v1,0%,0%,0%,0%
v2,0%,0%,0%,0%
v3,0%,100%,100%,100%


Strict valid_json is still the metric logged to MLflow; the table above only explains why it is 0%.


### 3.7 Final promotion justification (reviewed against the actual responses)
**Decision: Prompt v3 stays promoted to production, as a conditional promotion.**

**JSON validity.** Under the strict check logged in MLflow, no version passes: valid_json is 0% for v1, v2 and v3. All five v3 responses were wrapped in a ```json code fence even though the prompt forbids code fences. The table above shows the JSON inside the fence is well formed: after removing the fence, 5 of 5 v3 responses parse and match the schema (answer, confidence, key_points), while v1 and v2, which are plain prose, stay at 0%. v3 is therefore the only version that delivers machine-readable output, but it needs a fence-tolerant parser, constrained decoding or a larger model before it can be called production-safe.

**Quality.** The v3 answers are short, structured and mostly accurate. I read all five: four are correct, but the Q4 answer describes bias and variance the wrong way round, and the model still reported "high" confidence, so the confidence field cannot be trusted. v1 and v2 answers are longer prose; v1 on Q4 and v2 on Q3 reached the 256-token cap and may be cut off.

**Latency, tokens and cost.** v3 is the fastest at 63.69 s on average, against 92.71 s for v2 and 107.93 s for v1, because it generates fewer output tokens (121.0 against 193.8 and 221.0), even though its prompt is longer (103.6 input tokens against 69.6 and 47.6). It is also the cheapest at 0.00002246 USD per request (0.2246 USD per 10,000 requests), against 0.00002634 for v2 and 0.00002686 for v1. With strict validity tied at 0%, the selection rule fell through to latency, which picks v3. Because the alias is reversible, production can be pointed back to an earlier version in one call if v3 misbehaves.

In [20]:
print("Files written in this folder:")
for f in ["latency_tokens_cost.csv", "prompt_evaluation_results.csv", "mlflow.db"]:
    print(" ", f, "-", "found" if Path(f).exists() else "MISSING")
print("\nNext: open the MLflow UI (see README) and take the screenshots.")

Files written in this folder:
  latency_tokens_cost.csv - found
  prompt_evaluation_results.csv - found
  mlflow.db - found

Next: open the MLflow UI (see README) and take the screenshots.
